# Demo & Test

Sources :

- Prices : https://ember-energy.org/data/european-wholesale-electricity-price-data
- Reserves : https://www.services-rte.com/fr/telechargez-les-donnees-publiees-par-rte.html?category=market&type=balancing_capacity&subType=procured_reserves

## 1) Chargement et traitement de la data

J'ai gardé les transformations pour se rappeler de comment on a transformé la data depuis la source brute mais c'est un chantier.

-> A ne pas exécuter à nouveau, tout est dans le dossier data.

In [ ]:
PATH = r"C:\Users\thibc\Downloads"

import polars as pl
import pandas as pd
df_prices = pl.read_csv(f"{PATH}/France.csv")
df_reserves = pl.read_excel(f"{PATH}/Reserves.xlsx")

In [ ]:
df_prices = df_prices.to_pandas()
df_reserves = df_reserves.to_pandas()

In [ ]:
df_prices = df_prices[["Datetime (UTC)", "Price (EUR/MWhe)"]].rename(
    columns={"Datetime (UTC)": "Datetime", "Price (EUR/MWhe)": "Price"}
)
df_prices.index = pd.to_datetime(df_prices["Datetime"], utc=True)
df_prices.drop(columns=["Datetime"], inplace=True)
df_prices.to_parquet("data/prices.parquet")

In [ ]:
df = pd.read_parquet("../data/prices.parquet")
display(df)

In [ ]:
assert not df.index.duplicated().any()
pd.DataFrame(df).to_parquet("../data/prices.parquet")

In [ ]:
df = df["Price"].resample("15min").ffill()

In [ ]:
display(df)

In [ ]:
display(df_reserves)

In [ ]:
df = df_reserves.copy()

df["Date"] = pd.to_datetime(df["Date"])

start_hhmm = df["Heures"].str.split(" - ").str[0]

df["dt_local_naive"] = pd.to_datetime(
    df["Date"].dt.strftime("%Y-%m-%d") + " " + start_hhmm,
    format="%Y-%m-%d %H:%M",
    errors="raise",
)

df["dt_utc"] = (
    df["dt_local_naive"]
      .dt.tz_localize("Etc/GMT-1")
      .dt.tz_convert("UTC")
)

df = df.drop(columns=["dt_local_naive"])

In [ ]:
df_reserves = df.copy()
display(df_reserves)

In [ ]:
df_reserves.rename(columns={"Prix de la réserve (en euros/MW/15min)": "Price",
                          "Quantité contractualisée (en MW)": "Quantity",
                          "dt_utc": "Datetime"}, inplace=True)

In [ ]:
df_reserves.drop(columns=["Date", "Heures"], inplace=True)

In [ ]:
display(df_reserves)

In [ ]:
df_reserves["Way"] = df_reserves["Sens de la réserve"].map({
    "A la hausse": "UP",
    "A la baisse": "DOWN",
    "A la hausse et à la baisse": "UP_DOWN"
})

In [ ]:
df_reserves["Type"] = df_reserves["Type de réserve"].map({
    "Réserve primaire": "FCR",
    "Réserve secondaire": "aFRR",
})
df_reserves = df_reserves[df_reserves["Type"].isin(["FCR", "aFRR"])]
df_reserves = df_reserves[df_reserves["Type de produit"] == "STD"]

In [ ]:
df_reserves = df_reserves[df_reserves["Temporalité"] == "Journalier"]

In [ ]:
display(df_reserves)

In [ ]:
df_reserves = df_reserves[["Datetime", "Type", "Way", "Price"]]

In [ ]:
df_reserves.index = pd.to_datetime(df_reserves["Datetime"], utc=True)
df_reserves.drop(columns=["Datetime"], inplace=True)
display(df_reserves)

In [ ]:
df_reserves.to_parquet("data/reserves.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/reserves.parquet")
display(df)

In [ ]:
wide = (
    df.assign(col=df["Type"] + "_" + df["Way"])
      .pivot_table(index="Datetime", columns="col", values="Price", aggfunc="last")
      .sort_index()
)
wide = wide.rename(columns={
    "FCR_UP_DOWN": "FCR"
})

In [ ]:
wide = wide[wide.index.year == 2025]

In [ ]:
display(wide)

In [ ]:
wide.to_parquet("data/reserves.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/prices.parquet")
display(df)

In [ ]:
df.rename(columns={"Price": "Electricity"}, inplace=True)
df.to_parquet("data/prices.parquet")